> **Note:** This notebook requires a live OpenAI API key. Set `OPENAI_API_KEY` in your `.env` file before running. Smoke-run deferred — API key not available in CI.

# Vision Capabilities — Responses API

Send an image in, get text out. The **Responses API** accepts image content blocks inside `input=` using `{"type": "input_image", ...}`. We use `model="gpt-5.5"` for multimodal reasoning. No raw `requests` calls or manual headers needed.

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()

def get_response(prompt_question):
    response = client.responses.create(
        model="gpt-5.5",
        instructions="You are a helpful research and programming assistant",
        input=prompt_question,
    )
    return response.output_text


print(get_response("What is the capital of France?"))

## What if I want an IMAGE to go IN and TEXT to come OUT?

<img src="./assets-resources/attention-architecture.png" width=40%>

In [ ]:
import base64

# Encode a local image to a base64 data URL.
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

image_path = "./assets-resources/attention-architecture.png"
base64_image = encode_image(image_path)

response = client.responses.create(
    model="gpt-5.5",
    input=[
        {
            "role": "user",
            "content": [
                {
                    "type": "input_text",
                    "text": (
                        "Write me a caption for this image.\n"
                        "Output format:\n"
                        "'''\nCaption: <the contents of the caption>\n'''"
                    ),
                },
                {
                    "type": "input_image",
                    "image_url": f"data:image/png;base64,{base64_image}",
                    "detail": "auto",
                },
            ],
        }
    ],
)
captions_output = response.output_text
print(captions_output)

You can also pass a public image URL directly instead of base64:

In [ ]:
response = client.responses.create(
    model="gpt-5.5",
    input=[
        {
            "role": "user",
            "content": [
                {"type": "input_text", "text": "What is in this image?"},
                {
                    "type": "input_image",
                    "image_url": "https://upload.wikimedia.org/wikipedia/commons/4/47/PNG_transparency_demonstration_1.png",
                    "detail": "auto",
                },
            ],
        }
    ],
)
print(response.output_text)

In [ ]:
from IPython.display import Markdown
Markdown(captions_output)